# 12 Bridge | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: eksplozja klas przez dziedziczenie
2. 🌉 Uczestnicy: Abstraction i Implementor
3. 🔧 Implementacja z kompozycja
4. 🆚 Bridge vs Adapter
5. 🖥️ Zastosowania: cross-platform, sterowniki

## 1. 🔹 Problem: eksplozja klas przez dziedziczenie

Bridge (Most) to wzorzec strukturalny rozdzielajacy
abstrakcje od jej implementacji tak by obie mogly
byc zmieniane niezaleznie.

Problem: eksplozja klas gdy lacze dwa wymiary zmiennosci:
- Ksztalty (Circle, Rectangle, Triangle)
- Renderery (Vector, Raster, OpenGL)

Bez Bridge: 3 ksztalty x 3 renderery = 9 klas!
```
CircleSVG, CirclePNG, CircleOpenGL
RectSVG, RectPNG, RectOpenGL
TriangleSVG, TrianglePNG, TriangleOpenGL
```

Z Bridge: 3 + 3 = 6 klas!
```
Circle, Rectangle, Triangle (Abstraction)
SVGRenderer, PNGRenderer, OpenGLRenderer (Implementor)
```

Analogacja: pilot TV (Remote = Abstraction) + telewizor (TV = Implementor).
Mozna laczyc dowolny pilot z dowolnym telewizorem.

> 💡 Bridge rozdziela 'co robimy' (Abstraction) od
> 'jak robimy' (Implementor). Oba wymiary rosna niezaleznie.

In [ ]:
# Problem: eksplozja klas
class CircleSVG:
    def draw(self): print('Circle in SVG')

class CirclePNG:
    def draw(self): print('Circle as PNG')

class SquareSVG:
    def draw(self): print('Square in SVG')

class SquarePNG:
    def draw(self): print('Square as PNG')

# Ile klas potrzeba?
shapes = ['Circle', 'Square', 'Triangle', 'Line']
renderers = ['SVG', 'PNG', 'OpenGL', 'Canvas']
total = len(shapes) * len(renderers)
print(f'Ksztaltow: {len(shapes)}')
print(f'Rendererow: {len(renderers)}')
print(f'Klas bez Bridge: {total}')
print(f'Klas z Bridge: {len(shapes)} + {len(renderers)} = {len(shapes) + len(renderers)}')

# Dodanie nowego ksztaltu/renderera:
print('\nBez Bridge: dodac Triangle -> 4 nowe klasy (TriangleSVG, TrianglePNG, ...)')
print('Z Bridge: dodac Triangle -> 1 klasa (Triangle implementuje Shape)')

---

### 🐍 Cwiczenia - problem eksplozji

1. Policz ile klas potrzeba bez Bridge dla systemu platnosci
   z 4 metodami (karta, BLIK, przelew, gotowka) i 3 walutami
   (PLN, EUR, USD).
2. Narysuj tabele ksztalt x renderer z 5 ksztaltami i 5 rendererami.
   Podkresl jak rosnie problem.
3. *(Trudniejsze)* Zidentyfikuj dwa wymiary zmiennosci w systemie
   logowania: typy loggerow (debug, info, error) i handlery
   (console, file, network). Narysuj schemat Bridge.

In [ ]:
# Cwiczenie 1: platnosci
payment_methods = ['card', 'blik', 'transfer', 'cash']
currencies = ['PLN', 'EUR', 'USD']

without_bridge = len(payment_methods) * len(currencies)
with_bridge = len(payment_methods) + len(currencies)

print(f'Bez Bridge: {without_bridge} klas')
print(f'Z Bridge: {with_bridge} klas')
print(f'Oszczednosc: {without_bridge - with_bridge} klas')

In [ ]:
# Cwiczenie 2: tabela ksztalt x renderer
shapes5 = ['Circle', 'Square', 'Triangle', 'Line', 'Polygon']
renderers5 = ['SVG', 'PNG', 'OpenGL', 'Canvas', 'PDF']

print('Tabela klas bez Bridge:')
print(f'{'':15}', ' '.join(f'{r:8}' for r in renderers5))
for s in shapes5:
    print(f'{s:15}', ' '.join(f'{s+r:8}' for r in renderers5))
print(f'\nLaczna liczba: {len(shapes5) * len(renderers5)}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: schemat Bridge dla logowania
print('Bridge dla systemu logowania:')
print()
print('Abstraction (Logger):')
print('  - DebugLogger')
print('  - InfoLogger')
print('  - ErrorLogger')
print()
print('Implementor (LogHandler):')
print('  - ConsoleHandler')
print('  - FileHandler')
print('  - NetworkHandler')
print()
print('Kombinacje: 3 Loggers x 3 Handlers = 9 mozliwosci')
print('Klas: 3 + 3 = 6 (Bridge!) zamiast 9')

## 2. 🔹 Uczestnicy: Abstraction i Implementor

Struktura wzorca Bridge:

**Abstraction**:
- Definiuje interfejs wysokiego poziomu
- Trzyma referencje do Implementor (bridge)
- Deleguje do implementora nisko-poziomowe operacje

**RefinedAbstraction**:
- Rozszerza Abstraction o dodatkowe zachowania
- Nadal uzywa implementatora przez bridge

**Implementor**:
- Interfejs dla implementacji (moze sie roznic od Abstraction)
- Zazwyczaj zawiera bardziej prymitywne operacje

**ConcreteImplementor**:
- Konkretna implementacja Implementor
- Mozemy dodawac bez zmiany kodu Abstraction

Kluczowe: Abstraction **posiada** (HAS-A) Implementor,
nie dziedziczy (IS-A). To jest ten 'most' (bridge).

In [ ]:
from abc import ABC, abstractmethod

# Implementor: interfejs renderera
class Renderer(ABC):
    @abstractmethod
    def render_circle(self, x: int, y: int, r: int) -> None: ...
    @abstractmethod
    def render_rect(self, x: int, y: int, w: int, h: int) -> None: ...

# ConcreteImplementor: SVG
class VectorRenderer(Renderer):
    def render_circle(self, x, y, r) -> None:
        print(f'SVG: <circle cx="{x}" cy="{y}" r="{r}"/>')
    def render_rect(self, x, y, w, h) -> None:
        print(f'SVG: <rect x="{x}" y="{y}" width="{w}" height="{h}"/>')

# ConcreteImplementor: Raster
class RasterRenderer(Renderer):
    def render_circle(self, x, y, r) -> None:
        print(f'PNG: draw_circle({x}, {y}, r={r})')
    def render_rect(self, x, y, w, h) -> None:
        print(f'PNG: fill_rect({x}, {y}, {w}, {h})')

# Abstraction: ksztalt
class Shape(ABC):
    def __init__(self, renderer: Renderer):
        self.renderer = renderer  # <-- Bridge

    @abstractmethod
    def draw(self) -> None: ...
    @abstractmethod
    def resize(self, factor: float) -> 'Shape': ...

# RefinedAbstraction: Circle
class Circle(Shape):
    def __init__(self, renderer: Renderer, x: int, y: int, radius: int):
        super().__init__(renderer)
        self.x = x; self.y = y; self.radius = radius

    def draw(self) -> None:
        self.renderer.render_circle(self.x, self.y, self.radius)

    def resize(self, factor: float) -> 'Circle':
        self.radius = int(self.radius * factor)
        return self

# RefinedAbstraction: Rectangle
class Rectangle(Shape):
    def __init__(self, renderer: Renderer, x: int, y: int, w: int, h: int):
        super().__init__(renderer)
        self.x = x; self.y = y; self.w = w; self.h = h

    def draw(self) -> None:
        self.renderer.render_rect(self.x, self.y, self.w, self.h)

    def resize(self, factor: float) -> 'Rectangle':
        self.w = int(self.w * factor); self.h = int(self.h * factor)
        return self

# Klient laczy abstakcje z implementacja niezaleznie
for renderer in [VectorRenderer(), RasterRenderer()]:
    print(f'\n{type(renderer).__name__}:')
    shapes = [
        Circle(renderer, 10, 10, 50),
        Rectangle(renderer, 0, 0, 200, 100),
    ]
    for s in shapes:
        s.draw()

# Zmiana implementatora w runtime!
c = Circle(VectorRenderer(), 0, 0, 100)
c.draw()
c.renderer = RasterRenderer()  # swap renderer!
c.draw()

---

### 🐍 Cwiczenia - Abstraction / Implementor

1. Dodaj `CanvasRenderer` (implementacja dla HTML Canvas API).
   Przetestuj ze wszystkimi ksztaltami bez zmiany klas Shape.
2. Dodaj `Triangle(Shape)` (RefinedAbstraction) bez zmiany
   zadnego istniejacego Renderera.
3. *(Trudniejsze)* Napisz `MultiRenderer(renderers: list)` ktory
   deleguje do wielu rendererow jednoczesnie.

In [ ]:
# Cwiczenie 1: CanvasRenderer
class CanvasRenderer(Renderer):
    def render_circle(self, x, y, r) -> None:
        print(f'Canvas: ctx.arc({x}, {y}, {r}, 0, 2*Math.PI)')
    def render_rect(self, x, y, w, h) -> None:
        print(f'Canvas: ctx.fillRect({x}, {y}, {w}, {h})')

canvas = CanvasRenderer()
Circle(canvas, 50, 50, 30).draw()
Rectangle(canvas, 10, 10, 80, 40).draw()

In [ ]:
# Cwiczenie 2: Triangle
class Triangle(Shape):
    def __init__(self, renderer: Renderer, x1: int, y1: int,
                 x2: int, y2: int, x3: int, y3: int):
        super().__init__(renderer)
        self.points = [(x1, y1), (x2, y2), (x3, y3)]

    def draw(self) -> None:
        pts = ', '.join(f'({x},{y})' for x, y in self.points)
        print(f'{type(self.renderer).__name__}: triangle({pts})')

    def resize(self, factor: float) -> 'Triangle':
        self.points = [(int(x*factor), int(y*factor)) for x,y in self.points]
        return self

for r in [VectorRenderer(), RasterRenderer(), CanvasRenderer()]:
    Triangle(r, 0, 0, 50, 100, 100, 0).draw()

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: MultiRenderer
class MultiRenderer(Renderer):
    def __init__(self, *renderers: Renderer):
        self._renderers = renderers
    def render_circle(self, x, y, r) -> None:
        for renderer in self._renderers:
            renderer.render_circle(x, y, r)
    def render_rect(self, x, y, w, h) -> None:
        for renderer in self._renderers:
            renderer.render_rect(x, y, w, h)

multi = MultiRenderer(VectorRenderer(), RasterRenderer())
print('MultiRenderer:')
Circle(multi, 5, 5, 25).draw()
Rectangle(multi, 0, 0, 50, 30).draw()

## 3. 🔹 Implementacja z kompozycja

Bridge jest implementowany przez kompozycje - Abstraction
posiada pole wskazujace na Implementor.

Typowe wzorce implementacji:

1. Dependency injection w konstruktorze:
   `Shape(renderer: Renderer)` - wstrzykujemy przez init

2. Setter injection:
   `shape.renderer = new_renderer` - dynamiczna zmiana

3. Factory method:
   `@classmethod create_svg(cls)` - fabrycznie z konkretnym impl

4. Registry:
   `RENDERERS = {'svg': VectorRenderer, 'png': RasterRenderer}`

Zaleta kompozycji vs dziedziczenia:
- Runtime: mozemy zmienic implementacje w czasie wykonania
- Testowalnosc: wstrzykujemy mock w testach
- OCP: nowe implementacje bez zmiany abstrakcji

In [ ]:
# Pilot TV - przyklad z Bridge
from abc import ABC, abstractmethod

class Device(ABC):
    @abstractmethod
    def power_on(self) -> None: ...
    @abstractmethod
    def power_off(self) -> None: ...
    @abstractmethod
    def set_channel(self, ch: int) -> None: ...
    @abstractmethod
    def set_volume(self, vol: int) -> None: ...
    @abstractmethod
    def is_on(self) -> bool: ...
    @abstractmethod
    def get_channel(self) -> int: ...
    @abstractmethod
    def get_volume(self) -> int: ...

class TV(Device):
    def __init__(self): self._on = False; self._ch = 1; self._vol = 50
    def power_on(self) -> None: self._on = True; print('TV on')
    def power_off(self) -> None: self._on = False; print('TV off')
    def set_channel(self, ch: int) -> None: self._ch = ch; print(f'TV ch={ch}')
    def set_volume(self, vol: int) -> None: self._vol = max(0, min(100, vol)); print(f'TV vol={self._vol}')
    def is_on(self) -> bool: return self._on
    def get_channel(self) -> int: return self._ch
    def get_volume(self) -> int: return self._vol

class Radio(Device):
    def __init__(self): self._on = False; self._ch = 1; self._vol = 30
    def power_on(self) -> None: self._on = True; print('Radio on')
    def power_off(self) -> None: self._on = False; print('Radio off')
    def set_channel(self, ch: int) -> None: self._ch = ch; print(f'Radio freq={ch}.0 MHz')
    def set_volume(self, vol: int) -> None: self._vol = max(0, min(100, vol)); print(f'Radio vol={self._vol}')
    def is_on(self) -> bool: return self._on
    def get_channel(self) -> int: return self._ch
    def get_volume(self) -> int: return self._vol

# Abstraction: Remote
class Remote:
    def __init__(self, device: Device):
        self._device = device  # Bridge

    def toggle_power(self) -> None:
        if self._device.is_on():
            self._device.power_off()
        else:
            self._device.power_on()

    def channel_up(self) -> None:
        self._device.set_channel(self._device.get_channel() + 1)

    def channel_down(self) -> None:
        self._device.set_channel(max(1, self._device.get_channel() - 1))

    def volume_up(self) -> None:
        self._device.set_volume(self._device.get_volume() + 5)

    def volume_down(self) -> None:
        self._device.set_volume(self._device.get_volume() - 5)

# RefinedAbstraction: AdvancedRemote
class AdvancedRemote(Remote):
    def mute(self) -> None:
        self._device.set_volume(0)
        print('Muted')

    def go_to_channel(self, ch: int) -> None:
        self._device.set_channel(ch)

print('--- TV Remote ---')
tv = TV()
remote = AdvancedRemote(tv)
remote.toggle_power()
remote.channel_up()
remote.channel_up()
remote.volume_up()
remote.mute()

print('\n--- Radio Remote ---')
radio = Radio()
radio_remote = AdvancedRemote(radio)
radio_remote.toggle_power()
radio_remote.go_to_channel(105)
radio_remote.volume_up()

---

### 🐍 Cwiczenia - implementacja

1. Napisz `SmartRemote(Remote)` z metodami `record()` i
   `set_timer(minutes)`. Przetestuj z TV i Radio.
2. Napisz `DataExporter` (Implementor) z `export(data)`,
   `Report` (Abstraction). Zaimplementuj `CSVExporter` i `JSONExporter`.
3. *(Trudniejsze)* Napisz `Logger` (Abstraction) z `DebugLogger`,
   `InfoLogger`, `ErrorLogger` i handlery `ConsoleHandler`,
   `FileHandler`. Dodaj filtrowanie poziomu.

In [ ]:
# Cwiczenie 1: SmartRemote
class SmartRemote(AdvancedRemote):
    def record(self) -> None:
        if not self._device.is_on():
            self._device.power_on()
        print(f'Recording from channel {self._device.get_channel()}')

    def set_timer(self, minutes: int) -> None:
        print(f'Timer set: power off in {minutes} min')

smart = SmartRemote(TV())
smart.toggle_power()
smart.go_to_channel(7)
smart.record()
smart.set_timer(120)

In [ ]:
# Cwiczenie 2: DataExporter + Report
import json

class DataExporter(ABC):
    @abstractmethod
    def export(self, data: list) -> str: ...

class CSVExporter(DataExporter):
    def export(self, data: list) -> str:
        if not data: return ''
        header = ','.join(data[0].keys())
        rows = '\n'.join(','.join(str(v) for v in row.values()) for row in data)
        return f'{header}\n{rows}'

class JSONExporter(DataExporter):
    def export(self, data: list) -> str:
        return json.dumps(data, indent=2)

class Report:
    def __init__(self, exporter: DataExporter): self._exp = exporter
    def generate(self, title: str, data: list) -> str:
        header = f'=== {title} ===\n'
        return header + self._exp.export(data)

data = [{'name': 'Alice', 'score': 95}, {'name': 'Bob', 'score': 87}]
for exp in [CSVExporter(), JSONExporter()]:
    print(Report(exp).generate('Scores', data))
    print()

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: Logger Bridge
from datetime import datetime

class LogHandler(ABC):
    @abstractmethod
    def emit(self, level: str, msg: str) -> None: ...

class ConsoleHandler(LogHandler):
    def emit(self, level: str, msg: str) -> None:
        print(f'[{level}] {msg}')

class FileHandler(LogHandler):
    def __init__(self, path: str): self._path = path
    def emit(self, level: str, msg: str) -> None:
        print(f'FILE({self._path}): [{level}] {msg}')

LEVELS = {'DEBUG': 10, 'INFO': 20, 'WARNING': 30, 'ERROR': 40}

class Logger:
    def __init__(self, handler: LogHandler, min_level: str = 'DEBUG'):
        self._handler = handler
        self._min = LEVELS[min_level]

    def _log(self, level: str, msg: str) -> None:
        if LEVELS[level] >= self._min:
            ts = datetime.now().strftime('%H:%M:%S')
            self._handler.emit(level, f'{ts} {msg}')

    def debug(self, msg: str) -> None: self._log('DEBUG', msg)
    def info(self, msg: str) -> None: self._log('INFO', msg)
    def warning(self, msg: str) -> None: self._log('WARNING', msg)
    def error(self, msg: str) -> None: self._log('ERROR', msg)

info_log = Logger(ConsoleHandler(), min_level='INFO')
info_log.debug('debug msg')    # filtrowany
info_log.info('App started')   # widoczny
info_log.error('Crash!')

file_log = Logger(FileHandler('app.log'), min_level='WARNING')
file_log.info('info')           # filtrowany
file_log.warning('Low memory')
file_log.error('Disk full')

## 4. 🔹 Bridge vs Adapter

Bridge i Adapter wyglaja podobnie (oba uzyja kompozycji),
ale maja rozne intencje:

| Kryterium | Bridge | Adapter |
|---|---|---|
| Cel | Zaprojektowac niezaleznosc | Naprawic niezgodnosc |
| Czas | Z wyprzedzeniem (upfront) | Potem (retrofit) |
| Interfejsy | Oba projektowane razem | Jeden juz istnieje |
| Ilosc | 1 abstraction x wiele impl | 1 adapter na 1 adaptee |

**Bridge**: projektujemy oba interfejsy jednoczesnie,
zeby mozna bylo je zmieniac niezaleznie.

**Adapter**: mamy juz istniejace interfejsy i chcemy
sprawic by wspolpracowaly.

> 💡 Mnemonic: Adapter to 'plastry na rany' (retrofit),
> Bridge to 'zaplanowany most' (upfront design).

In [ ]:
# Porownanie: Bridge vs Adapter

# ADAPTER: mamy istniejace klasy i chcemy spojnosc
class OldPaymentSystem:
    def charge_card(self, card: str, amount_cents: int) -> bool:
        print(f'Old: charge {amount_cents} cents on {card}')
        return True

class PaymentInterface:  # interfejs oczekiwany przez nowy kod
    def pay(self, amount: float) -> bool: ...

class PaymentAdapter(PaymentInterface):
    def __init__(self, old: OldPaymentSystem, card: str):
        self._old = old; self._card = card
    def pay(self, amount: float) -> bool:
        return self._old.charge_card(self._card, int(amount * 100))

# BRIDGE: projektujemy od poczatku z niezaleznoscia
class PaymentProcessor(ABC):  # Implementor - zaprojektowany razem
    @abstractmethod
    def process(self, amount: float, currency: str) -> bool: ...

class StripeProcessor(PaymentProcessor):
    def process(self, amount: float, currency: str) -> bool:
        print(f'Stripe: {amount} {currency}')
        return True

class PayPalProcessor(PaymentProcessor):
    def process(self, amount: float, currency: str) -> bool:
        print(f'PayPal: {amount} {currency}')
        return True

class Order:  # Abstraction - zaprojektowane razem
    def __init__(self, processor: PaymentProcessor):
        self._processor = processor  # Bridge
    def checkout(self, amount: float, currency: str = 'PLN') -> bool:
        print(f'Checkout {amount} {currency}')
        return self._processor.process(amount, currency)

class SubscriptionOrder(Order):
    def renew(self, monthly_fee: float) -> bool:
        print('Renewing subscription')
        return self._processor.process(monthly_fee, 'EUR')

print('Adapter:')
PaymentAdapter(OldPaymentSystem(), '4111..').pay(49.99)

print('\nBridge:')
Order(StripeProcessor()).checkout(99.99)
SubscriptionOrder(PayPalProcessor()).renew(19.99)

---

### 🐍 Cwiczenia - Bridge vs Adapter

1. Masz `LegacyDB.execute(sql)` i chcesz interfejs `Database.query(sql)`.
   Uzyj Adapter. Teraz zaprojektuj od zera Bridge dla dwoch baz.
2. Podaj 3 przyklady z bibliotek Pythona gdzie uzyto Bridge.
3. *(Trudniejsze)* Przeksztalc istniejacy Adapter na Bridge
   - co sie zmienia w projekcie?

In [ ]:
# Cwiczenie 1: LegacyDB Adapter, potem Bridge

# Adapter: naprawiamy istniejacy kod
class LegacyDB:
    def execute(self, sql: str) -> list:
        print(f'Legacy SQL: {sql}')
        return [{'id': 1}]

class DatabaseAdapter:
    def __init__(self, legacy: LegacyDB): self._db = legacy
    def query(self, sql: str) -> list: return self._db.execute(sql)

print('Adapter:')
DatabaseAdapter(LegacyDB()).query('SELECT * FROM users')

# Bridge: od zera, dwa wymiary niezalezne
class DBBackend(ABC):
    @abstractmethod
    def run(self, sql: str) -> list: ...

class SQLiteBackend(DBBackend):
    def run(self, sql: str) -> list:
        print(f'SQLite: {sql}'); return [{'id': 1}]

class PostgresBackend(DBBackend):
    def run(self, sql: str) -> list:
        print(f'Postgres: {sql}'); return [{'id': 1}]

class Database2:
    def __init__(self, backend: DBBackend): self._backend = backend
    def query(self, sql: str) -> list: return self._backend.run(sql)
    def get_users(self) -> list: return self.query('SELECT * FROM users')

class ReadOnlyDatabase(Database2):
    def query(self, sql: str) -> list:
        if sql.strip().upper().startswith(('INSERT', 'UPDATE', 'DELETE')):
            raise PermissionError('Read-only!')
        return self._backend.run(sql)

print('\nBridge:')
Database2(SQLiteBackend()).get_users()
Database2(PostgresBackend()).get_users()
try:
    ReadOnlyDatabase(SQLiteBackend()).query('DELETE FROM users')
except PermissionError as e:
    print(f'Denied: {e}')

In [ ]:
# Cwiczenie 2: Bridge w bibliotekach Pythona
examples = [
    'logging module: Logger (Abstraction) + Handler (Implementor)',
    'matplotlib: Figure (Abstraction) + Backend (Implementor: Agg, Qt, Web)',
    'SQLAlchemy: ORM (Abstraction) + Dialect (Implementor: mysql, postgres, sqlite)',
]
for e in examples:
    print(f'- {e}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: Adapter -> Bridge
print('Przeksztalcenie Adaptera na Bridge:')
print()
print('PRZED (Adapter - reaktywny):')
print('  - OldPaymentSystem juz istnieje')
print('  - PaymentAdapter opakowuje konkrena klase')
print('  - Trudno dodac nowe implementacje')
print()
print('PO (Bridge - proaktywny):')
print('  - PaymentProcessor (interfejs Implementor) zaprojektowany razem')
print('  - Order (Abstraction) zalezy od interfejsu, nie konkretu')
print('  - Dodanie nowego procesora: tylko nowa klasa implementujaca PaymentProcessor')
print()
print('Kluczowa zmiana: zaleznosc od INTERFEJSU zamiast od KLASY')

## 5. 🔹 Zastosowania: cross-platform, sterowniki

Bridge doskonale sprawdza sie w systemach cross-platform:

**GUI Toolkits**:
- Abstraction: Button, TextBox, Menu (abstrakcja UI)
- Implementor: Win32API, Cocoa, GTK (platforma systemowa)
- Jedna implementacja UI dziala na wielu platformach

**Sterowniki urzadzen**:
- Abstraction: PrintDocument, ScanDocument
- Implementor: HPDriver, EpsonDriver, CanonDriver
- Aplikacja nie zmienia sie przy zmianie sterownika

**Bazy danych (ORMs)**:
- Abstraction: Model, Query, Session
- Implementor: MySQLDialect, PostgresDialect, SQLiteDialect

**Python logging**:
- Abstraction: Logger, LogRecord
- Implementor: StreamHandler, FileHandler, SMTPHandler

> 💡 Jesli widzisz `setHandler()` lub `setBackend()` w API -
> prawdopodobnie masz do czynienia z Bridge.

In [ ]:
# Bridge w stylu logging module
import sys
from abc import ABC, abstractmethod

class LogHandler(ABC):
    @abstractmethod
    def emit(self, record: dict) -> None: ...

class StreamHandler(LogHandler):
    def __init__(self, stream=None):
        self._stream = stream or sys.stdout
    def emit(self, record: dict) -> None:
        self._stream.write(f"{record['level']}: {record['msg']}\n")

class FileLogHandler(LogHandler):
    def __init__(self, path: str): self._path = path
    def emit(self, record: dict) -> None:
        print(f'[FILE:{self._path}] {record["level"]}: {record["msg"]}')

class MemoryHandler(LogHandler):
    def __init__(self): self.records: list[dict] = []
    def emit(self, record: dict) -> None: self.records.append(record)

LEVELS = {'DEBUG': 10, 'INFO': 20, 'WARNING': 30, 'ERROR': 40, 'CRITICAL': 50}

class AppLogger:
    def __init__(self, name: str, level: str = 'DEBUG'):
        self.name = name
        self._level = LEVELS[level]
        self._handlers: list[LogHandler] = []

    def add_handler(self, handler: LogHandler) -> 'AppLogger':
        self._handlers.append(handler)
        return self

    def _log(self, level: str, msg: str) -> None:
        if LEVELS.get(level, 0) >= self._level:
            record = {'level': level, 'logger': self.name, 'msg': msg}
            for h in self._handlers:
                h.emit(record)

    def debug(self, msg: str) -> None: self._log('DEBUG', msg)
    def info(self, msg: str) -> None: self._log('INFO', msg)
    def warning(self, msg: str) -> None: self._log('WARNING', msg)
    def error(self, msg: str) -> None: self._log('ERROR', msg)

# Konfiguracja: wiele handlerow jednoczesnie (jak logging.addHandler)
memory = MemoryHandler()
logger = (AppLogger('myapp', 'INFO')
    .add_handler(StreamHandler())
    .add_handler(FileLogHandler('app.log'))
    .add_handler(memory))

logger.debug('ignored')    # ponizej INFO
logger.info('started')
logger.warning('memory')
logger.error('crash!')

print(f'\nIn memory: {len(memory.records)} records')
for r in memory.records:
    print(f'  {r}')

---

### 🐍 Cwiczenia - zastosowania

1. Napisz `NotificationSystem` (Bridge) z kanalami `EmailChannel`,
   `SMSChannel` i typami `AlertNotification`, `InfoNotification`.
2. Napisz cross-platform `UIButton` z platformami `Windows`, `MacOS`,
   `Linux` - metoda `render() -> str`.
3. *(Trudniejsze)* Zaprojektuj Bridge dla parsera plikow:
   `ParserAbstraction` (CSV, JSON, XML) i `StorageBackend`
   (MemoryStorage, FileStorage, DatabaseStorage).

In [ ]:
# Cwiczenie 1: NotificationSystem Bridge
class MessageChannel(ABC):
    @abstractmethod
    def send(self, recipient: str, message: str) -> None: ...

class EmailChannel(MessageChannel):
    def send(self, recipient: str, message: str) -> None:
        print(f'Email to {recipient}: {message}')

class SMSChannel(MessageChannel):
    def send(self, recipient: str, message: str) -> None:
        print(f'SMS to {recipient}: {message[:160]}')

class Notification(ABC):
    def __init__(self, channel: MessageChannel): self._ch = channel
    @abstractmethod
    def notify(self, user: str, event: str) -> None: ...

class AlertNotification(Notification):
    def notify(self, user: str, event: str) -> None:
        self._ch.send(user, f'ALERT: {event}')

class InfoNotification(Notification):
    def notify(self, user: str, event: str) -> None:
        self._ch.send(user, f'INFO: {event}')

alerts = [
    AlertNotification(EmailChannel()),
    AlertNotification(SMSChannel()),
    InfoNotification(EmailChannel()),
]
for n in alerts:
    n.notify('alice@x.com', 'Login from new device')

In [ ]:
# Cwiczenie 2: cross-platform UIButton
class PlatformAPI(ABC):
    @abstractmethod
    def draw_button(self, label: str, w: int, h: int) -> str: ...

class WindowsAPI(PlatformAPI):
    def draw_button(self, label: str, w: int, h: int) -> str:
        return f'Win32: [{label}] ({w}x{h})'

class MacOSAPI(PlatformAPI):
    def draw_button(self, label: str, w: int, h: int) -> str:
        return f'Cocoa: <{label}> ({w}x{h})'

class LinuxAPI(PlatformAPI):
    def draw_button(self, label: str, w: int, h: int) -> str:
        return f'GTK: button({label}) ({w}x{h})'

class UIButton:
    def __init__(self, label: str, platform: PlatformAPI):
        self.label = label
        self._platform = platform
    def render(self) -> str:
        return self._platform.draw_button(self.label, 120, 40)

for platform in [WindowsAPI(), MacOSAPI(), LinuxAPI()]:
    btn = UIButton('Submit', platform)
    print(btn.render())

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: Parser Bridge
import json

class StorageBackend(ABC):
    @abstractmethod
    def save(self, data: list) -> None: ...
    @abstractmethod
    def load(self) -> list: ...

class MemoryStorage(StorageBackend):
    def __init__(self): self._data = []
    def save(self, data: list) -> None: self._data = data[:]; print(f'Memory: saved {len(data)} items')
    def load(self) -> list: return self._data

class FileStorage(StorageBackend):
    def __init__(self, path: str): self._path = path
    def save(self, data: list) -> None: print(f'File({self._path}): saved {len(data)} items')
    def load(self) -> list: return []

class DataParser(ABC):
    def __init__(self, storage: StorageBackend): self._storage = storage
    @abstractmethod
    def parse(self, raw: str) -> list: ...
    def process(self, raw: str) -> list:
        data = self.parse(raw)
        self._storage.save(data)
        return data

class CSVParser(DataParser):
    def parse(self, raw: str) -> list:
        lines = raw.strip().split('\n')
        headers = lines[0].split(',')
        return [dict(zip(headers, row.split(','))) for row in lines[1:]]

class JSONParser(DataParser):
    def parse(self, raw: str) -> list:
        return json.loads(raw)

csv_data = 'name,age\nAlice,30\nBob,25'
json_data = '[{"name": "Charlie", "age": 35}]'

CSVParser(MemoryStorage()).process(csv_data)
JSONParser(FileStorage('out.json')).process(json_data)